In [1]:
import pandas as pd                  #load csv files
import numpy as np                   #mathematic manipulations

from sklearn.model_selection import train_test_split     #data splitting

from sklearn.preprocessing import OneHotEncoder      #data preprocessing

from sklearn.compose import ColumnTransformer        #data preprocessing

from sklearn.pipeline import Pipeline                #data preprocessing

from sklearn.ensemble import RandomForestRegressor   #machine learning model

from sklearn.multioutput import MultiOutputRegressor #machine learning model

from sklearn.metrics import (                       #evaluation metrics
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import joblib                           #save model

Load Dataset

In [2]:
df = pd.read_csv("../data/processed/training_dataset.csv")

In [3]:
df.head()

,age,gender,height_m,weight_kg,bmi,health_condition,activity_level,diet_type,allergy,target_calories,target_protein,target_carbohydrates,target_fat,target_fiber,target_max_sugar,target_max_sodium
0,71,Female,1.61,112,26.5,Diabetes,Low,Non-Vegetarian,No Allergy,1400,90,155,55,35,25,1800
1,56,Female,1.79,111,19.1,Healthy,Medium,Non-Vegetarian,No Allergy,2200,75,275,70,30,50,2300
2,34,Female,1.72,96,19.3,Healthy,Low,Vegan,Dairy,2000,75,260,70,30,50,2300
3,79,Female,1.68,114,25.9,Healthy,Medium,Vegetarian,Dairy,1950,75,265,70,30,50,2300
4,67,Male,1.65,47,19.8,Hypertension,Low,Non-Vegetarian,Seafood,1650,80,225,60,35,40,1500


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   age                   20000 non-null  int64  
 1   gender                20000 non-null  str    
 2   height_m              20000 non-null  float64
 3   weight_kg             20000 non-null  int64  
 4   bmi                   20000 non-null  float64
 5   health_condition      20000 non-null  str    
 6   activity_level        20000 non-null  str    
 7   diet_type             20000 non-null  str    
 8   allergy               20000 non-null  str    
 9   target_calories       20000 non-null  int64  
 10  target_protein        20000 non-null  int64  
 11  target_carbohydrates  20000 non-null  int64  
 12  target_fat            20000 non-null  int64  
 13  target_fiber          20000 non-null  int64  
 14  target_max_sugar      20000 non-null  int64  
 15  target_max_sodium     20000 no

In [5]:
df.isnull().sum()

age                     0
gender                  0
height_m                0
weight_kg               0
bmi                     0
health_condition        0
activity_level          0
diet_type               0
allergy                 0
target_calories         0
target_protein          0
target_carbohydrates    0
target_fat              0
target_fiber            0
target_max_sugar        0
target_max_sodium       0
dtype: int64

| Check                                   | 
| --------------------------------------- | 
| No missing values                       |
| Correct data types                      |
| Features separated from targets         | 
| 20,000 samples                          | 
| Multiple outputs                        | 
| Mix of numerical & categorical features | 


This is a solid dataset for training ML.

What are Features?

Features are the information we already know about the patient.

From this information, we want the model to predict nutrition requirements.

These are called independent variables or input variables.

What are Targets?

Targets are what we want the model to learn.

These are called dependent variables or labels.

Input Features (X)

In [6]:
X = df[
    [
        "age",
        "gender",
        "height_m",
        "weight_kg",
        "bmi",
        "health_condition",
        "activity_level",
        "diet_type",
        "allergy"
    ]
]

Target Variables (y)

In [7]:
y = df[
    [
        "target_calories",
        "target_protein",
        "target_carbohydrates",
        "target_fat",
        "target_fiber",
        "target_max_sugar",
        "target_max_sodium"
    ]
]

In [8]:
print("Features Shape :", X.shape)
print("Targets Shape  :", y.shape)

Features Shape : (20000, 9)
Targets Shape  : (20000, 7)


Split the ML_training_dataset into two parts one for testing and other for testing.

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

test_size=0.2 ==> 20% of the data is reserved for testing.

The model never sees those 4,000 patients during training.

random_state=42 ===> Imagine it like shuffling deck of cards. This constant 42 makes the split reproducible.

There is no mathematical reason, for why only 42?. It's just a widely used convention in the ML community

In [10]:
print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)

print("y_train :", y_train.shape)
print("y_test  :", y_test.shape)

X_train : (16000, 9)
X_test  : (4000, 9)
y_train : (16000, 7)
y_test  : (4000, 7)


Machine learning algorithms cannot work directly with text.

Let's separate our columns into:

Numerical features

Categorical features

In [11]:
categorical_features = [
    "gender",
    "health_condition",
    "activity_level",
    "diet_type",
    "allergy"
]

numerical_features = [
    "age",
    "height_m",
    "weight_kg",
    "bmi"
]

Use one-hot encoding and convert the categorical_features into numericals.

Machines only understand only numbers but not strings.

In [12]:
print(X_train.shape)
print(X_test.shape)

print(y_train.shape)
print(y_test.shape)

print(categorical_features)
print(numerical_features)

(16000, 9)
(4000, 9)
(16000, 7)
(4000, 7)
['gender', 'health_condition', 'activity_level', 'diet_type', 'allergy']
['age', 'height_m', 'weight_kg', 'bmi']


Why do we need preprocessing?

The model understands

45 the age

But it cannot understand

Female

Diabetes

Vegetarian

We must convert them into numbers.

Why not do it manually?

It works...

Until deployment.

Suppose tomorrow a new user enters new value.

API crashes because the training columns and prediction columns no longer match.

A Pipeline prevents this problem.

What is a Pipeline?


1. Patient Data
      
2. One-Hot Encoding

3. Random Forest

4. Nutrition Prediction

Instead of manually doing each step every time,

the Pipeline remembers the complete workflow.

In [13]:
preprocessor = ColumnTransformer(                    #data preprocessor
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ],
    remainder="passthrough"
)

ColumnTransformer => only applies encoding to string columns not to the numerical columns.

transformers => A ColumnTransformer can have many transformers. We currently have only one.

"categorial" => This is only name. We can give anything.

OneHotEncoder => Used to convert strings to numbers.

unknown strings are ignored => After deploying, if user enters some new value, that is ignored by onehotencoder instead of crashing.

categorial_features => Only this set of features are strings and they need to be encoded so pass them

remainder => makes sure that every feature is encoded into numericals. They are multiple features needed to encoded. So, this is a small check.

Build the Machine Learning Model

In [14]:
model = MultiOutputRegressor(
    RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )
)

Why are we wrapping the Random Forest?

Normally, Random Forest predicts, One value.

Our model need to predict seven values.

MultiOutputRegressor internally trains one Random Forest for each target.

Each forest specializes in one nutrition target like forest 1 in calories, forest 2 in fats, forest 3 in protiens and so on..

Why n_estimators=200?

A Random Forest is made of many decision trees.

Each tree votes.

More trees generally improve stability, but increase training time.

For 20,000 samples, 200 is a good balance between performance and speed.

Why n_jobs=-1?

Random Forest can train trees in parallel.

n_jobs=-1

Use all available CPU cores.

This makes training significantly faster on modern computers.

Build the Complete Pipeline

In [15]:
pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ]
)

In [16]:
print(pipeline)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['gender', 'health_condition',
                                                   'activity_level',
                                                   'diet_type', 'allergy'])])),
                ('model',
                 MultiOutputRegressor(estimator=RandomForestRegressor(n_estimators=200,
                                                                      n_jobs=-1,
                                                                      random_state=42)))])


Train the Model

In [17]:
print("Training started...")

pipeline.fit(X_train, y_train)

print("Training completed!")

Training started...
Training completed!


You only wrote one line:

pipeline.fit(X_train, y_train)

But internally, Scikit-learn performs several steps.

1. Training Data

2. ColumnTransformer
      
3. OneHotEncoder
      
4. Numerical + Encoded Features
      
5. MultiOutputRegressor
      
6. Random Forest #1 → Calories

Random Forest #2 → Protein

Random Forest #3 → Carbohydrates

Random Forest #4 → Fat

Random Forest #5 → Fiber

Random Forest #6 → Sugar

Random Forest #7 → Sodium

So although it looks like one model, internally 7 Random Forest models are trained.

Make Predictions

In [18]:
y_pred = pipeline.predict(X_test)

What is y_pred?

It is the model's prediction for every patient in the test set.

Actual nutrition:

Calories : 2000

Protein  : 95

Fat      : 55

Model predicts:

Calories : 1985

Protein  : 94

Fat      : 57

That entire prediction is stored in y_pred

In [19]:
print(y_pred.shape)

(4000, 7)


Compare Predictions

In [20]:
prediction_df = pd.DataFrame(
    y_pred,
    columns=y.columns
)

prediction_df.head()

,target_calories,target_protein,target_carbohydrates,target_fat,target_fiber,target_max_sugar,target_max_sodium
0,2350.0,85.0,295.0,75.0,30.0,50.0,2300.0
1,1450.0,100.0,150.0,45.0,40.0,20.0,1800.0
2,2000.0,75.0,260.0,70.0,30.0,50.0,2300.0
3,2950.0,105.0,350.0,80.0,30.0,60.0,2300.0
4,1400.0,95.0,165.0,50.0,40.0,25.0,1800.0


Now compare with the actual values:

In [21]:
comparison = pd.concat(
    [
        y_test.reset_index(drop=True),
        prediction_df
    ],
    axis=1,
    keys=["Actual", "Predicted"]
)

comparison.head(10)

Actual                                                              \
  target_calories target_protein target_carbohydrates target_fat target_fiber   
0            2350             85                  295         75           30   
1            1450            100                  150         45           40   
2            2000             75                  260         70           30   
3            2950            105                  350         80           30   
4            1400             95                  165         50           40   
5            2600             85                  295         75           30   
6            2600             85                  295         75           30   
7            2300             75                  275         70           30   
8            1750             75                  250         70           30   
9            2500             85                  295         75           30   

                                           Predicted                 \
  target_max_sugar target_max_sodium target_calories target_protein   
0               50              2300          2350.0           85.0   
1               20              1800          1450.0          100.0   
2               50              2300          2000.0           75.0   
3               60              2300          2950.0          105.0   
4               25              1800          1400.0           95.0   
5               50              2300          2600.0           85.0   
6               50              2300          2600.0           85.0   
7               50              2300          2300.0           75.0   
8               50              2300          1750.0           75.0   
9               50              2300          2500.0           85.0   

                                                                 \
  target_carbohydrates target_fat target_fiber target_max_sugar   
0                295.0       75.0         30.0             50.0   
1                150.0       45.0         40.0             20.0   
2                260.0       70.0         30.0             50.0   
3                350.0       80.0         30.0             60.0   
4                165.0       50.0         40.0             25.0   
5                295.0       75.0         30.0             50.0   
6                295.0       75.0         30.0             50.0   
7                275.0       70.0         30.0             50.0   
8                250.0       70.0         30.0             50.0   
9                295.0       75.0         30.0             50.0   

                     
  target_max_sodium  
0            2300.0  
1            1800.0  
2            2300.0  
3            2300.0  
4            1800.0  
5            2300.0  
6            2300.0  
7            2300.0  
8            2300.0  
9            2300.0

Evaluate the Model

calculate three metrics.

In [22]:
mae = mean_absolute_error(y_test, y_pred)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))

r2 = r2_score(y_test, y_pred)

print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"R²   : {r2:.4f}")

MAE  : 0.06
RMSE : 1.87
R²   : 1.0000


Understanding the Metrics

MAE (Mean Absolute Error)

Suppose:

Actual calories:

2000

Prediction:

1985

Error:

15 calories

MAE is the average error across all predictions.

Lower is better.

RMSE (Root Mean Squared Error)

RMSE penalizes large mistakes more heavily than MAE.

If the model makes one very bad prediction, RMSE increases significantly.

Lower is better.

R² Score

1.00  → Perfect prediction

0.90  → Excellent

0.80  → Very good

0.60  → Acceptable

0.00  → No learning

For this synthetic dataset, because the nutrition targets are generated from rules, it's quite possible to achieve a very high R².